# M2 – Mẫu pilot TẠM (Chicago Taxi Trips, 1 tuần)

**Mẫu tạm của M2** để chạy thử K-Means và DBSCAN khi chưa có dữ liệu của M1. Không phải dữ liệu chính thức, không dùng cho kết quả cuối.

Chạy trên **Google Colab**: cổng dữ liệu Chicago trả lỗi 403 khi truy cập từ máy ở Việt Nam (15/11/2026).

Mở trên Colab: https://colab.research.google.com/github/tanbentam/KhaiPhaDuLieu/blob/main/notebooks/m2_pilot_tam.ipynb

Các bước:
1. Đối chiếu tên cột với metadata của cổng dữ liệu.
2. Đếm số chuyến trên server, tải theo trang, đối chiếu lại số dòng.
3. Làm sạch tối thiểu, gán `time_slice`, chiếu sang EPSG:26916.
4. Lưu `pilot_m2_v0.parquet` và manifest, rồi tải về máy.

Sau khi tải về: đặt 2 file vào `data/pilot_m2/` trong repo trên máy. Thư mục `data/` đã nằm trong `.gitignore`, không bị commit.

In [ ]:
# ===== O THIET LAP (chep nguyen o nay vao dau moi notebook) =====
# Colab: clone repo, cai thu vien, them goc repo vao sys.path.
# May ca nhan: tu tim goc repo, khong cai gi them.
import os, sys, subprocess, pathlib

REPO_URL = "https://github.com/tanbentam/KhaiPhaDuLieu.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = pathlib.Path("/content/KhaiPhaDuLieu")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements.txt")], check=True)
    os.chdir(ROOT)
else:
    here = pathlib.Path.cwd().resolve()
    ROOT = next(p for p in (here, *here.parents) if (p / "config" / "experiment.yaml").is_file())

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Du lieu that tren Colab: bo comment 2 dong duoi khi nhom da chot thu muc Drive
# from src.colab import link_drive_data
# link_drive_data("KhaiPhaDuLieu")

print("Colab:", IN_COLAB, "| Goc repo:", ROOT)

## 1. Tham số

Tuần 17–23/04/2023 (thứ Hai đến Chủ nhật): không có ngày lễ, không đổi giờ mùa hè, có đủ 3 lát cắt.

In [ ]:
from src.config import load_config, resolve_path
from src import m2_pilot as P

cfg = load_config()
START, END = "2023-04-17", "2023-04-23"
OUT_PARQUET = resolve_path(cfg, "pilot_m2")
OUT_MANIFEST = resolve_path(cfg, "pilot_m2_manifest")
print("Luu vao:", OUT_PARQUET)

## 2. Tải và làm sạch

Mỗi trang 50.000 dòng. Hàm dừng khi gặp trang rỗng và báo lỗi nếu số dòng tải về khác số dòng server đếm được.

In [ ]:
man = P.build_pilot(START, END, cfg["time_slices"], OUT_PARQUET, OUT_MANIFEST,
                    seed=cfg["project"]["seed"], crs=cfg["project"]["crs"])
print()
print("Server dem   :", f"{man['rows_server_count']:,}")
print("Sau lam sach :", f"{man['rows_after_cleaning']:,}")
print("Toa do duy nhat:", f"{man['unique_pickup_coords']:,}", "(du lieu centroid -> rat it diem phan biet)")
print("Theo lat cat :", man["rows_by_time_slice"])
import pandas as pd
pd.DataFrame(man["cleaning_steps"])

## 3. Kiểm tra nhanh

In [ ]:
df = pd.read_parquet(OUT_PARQUET)
print(df.dtypes.to_string())
print()
print("x_m:", f"{df.x_m.min():,.0f} .. {df.x_m.max():,.0f}", "| y_m:", f"{df.y_m.min():,.0f} .. {df.y_m.max():,.0f}")
print("Ngay:", df.trip_start_timestamp.min(), "..", df.trip_start_timestamp.max())
print("Co chat luong:", {c: int(df[c].sum()) for c in df.columns if c.startswith("flag_")})
assert df.trip_id.is_unique and df[["x_m", "y_m"]].notna().all().all()
df.head()

## 4. Tải file về máy

Tải 2 file rồi đặt vào `D:\\3\\KhaiPhaDuLieu\\data\\pilot_m2\\` (hoặc `data/pilot_m2/` trong repo của bạn).

In [ ]:
if IN_COLAB:
    from google.colab import files
    files.download(str(OUT_PARQUET))
    files.download(str(OUT_MANIFEST))
    # Hoac luu len Drive (khi nhom da chot thu muc):
    # from google.colab import drive; drive.mount("/content/drive")
    # import shutil; dst = pathlib.Path("/content/drive/MyDrive/KhaiPhaDuLieu/data/pilot_m2"); dst.mkdir(parents=True, exist_ok=True)
    # shutil.copy(OUT_PARQUET, dst); shutil.copy(OUT_MANIFEST, dst)
else:
    print("Dang chay tren may ca nhan; file da nam o", OUT_PARQUET.parent)